# EXP-001 — Kiểm tra: train trên 25% ảnh ngẫu nhiên (seed 1)
Trọng số mềm và lọc cứng 25% theo AU đều cho Δ ≈ 0. Câu hỏi: **head có nhạy với dữ liệu train không?**

Train trên **25% ảnh CAER-S chọn ngẫu nhiên** (không dùng AU, không trọng số), 20 epoch để **cùng số bước cập nhật** với 5 epoch đủ dữ liệu.
- Kết quả vẫn ≈ EXP-000 → head không nhạy với việc chọn ảnh; lọc/đổi trọng số theo ảnh (EXP-001, EXP-002) khó có tác dụng.
- Kết quả tụt rõ → dữ liệu có ảnh hưởng; lọc theo AU chỉ là chưa chọn đúng ảnh.

Không cần precompute hay py-feat. Thời gian: cài đặt ~10 phút, tải FlanT5, train ~45 phút (BLIP-2 vẫn chạy mỗi bước), test.

# Cấu hình

In [1]:
SEED = 1
FRAC = 0.25                      # giữ 25% ảnh, chọn ngẫu nhiên (không dùng AU)
EPOCHS = 20                      # 4× epoch → cùng số bước cập nhật (~480) như train đủ dữ liệu 5 epoch
JOB_ID = f"exp001_rand{round(FRAC*100)}_l14_seed{SEED}"
CMP_JOBS = {"EXP-001 soft seed1": "exp001_au_l14_seed1", "EXP-001 drop25 seed1": "exp001_au_drop25_l14_seed1"}
BRANCH = "exp/001-au-detector-filtering"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
SUBSET_DATA = f"/content/CAER-S_rand{round(FRAC*100)}/train"   # symlink tới ảnh gốc
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/train"    # checkpoint, log, kết quả test
T5_CACHE = "/root/.cache/huggingface/hub/models--google--flan-t5-xxl"

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

def hf_offline():
    # FlanT5 đã có trong cache thì không gọi HuggingFace nữa
    os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

# Mount Drive & GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
# BLIP-2 FlanT5-XXL cần ~24GB chỉ để nạp model → GPU 22–24GB (L4, A10) sẽ hết bộ nhớ
gpu_mem = int(subprocess.check_output("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True).split()[0])
assert gpu_mem >= 35000, f"GPU chỉ có {gpu_mem} MiB — đổi runtime sang A100 (Runtime > Change runtime type)"

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


# Clone repo (nhánh EXP-001)

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (374/374), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 374 (delta 277), reused 355 (delta 259), pack-reused 0 (from 0)
Receiving objects: 100% (374/374), 30.67 MiB | 19.46 MiB/s, done.
Resolving deltas: 100% (277/277), done.
/content/Exp-CLIP
Already on 'exp/001-au-detector-filtering'
Your branch is up to date with 'origin/exp/001-au-detector-filtering'.
Already up to date.
15d4084 (HEAD -> exp/001-au-detector-filtering, origin/exp/001-au-detector-filtering) EXP-001: add --drop hard filter to weight stage


# Giải nén dữ liệu (train + test)

In [4]:
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Cài dependencies (venv310)\ntorch 2.0.1 + lavis: dùng cho target, proto, weight, train, test.

In [5]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Install

# Tạo tập con ngẫu nhiên
Symlink, không copy ảnh. Chọn ngẫu nhiên trên toàn bộ ảnh (cố định `random.Random(0)`), không theo nhãn; in tỉ lệ từng lớp để kiểm tra.

In [6]:
import glob, random, shutil, collections
all_imgs = sorted(glob.glob(f"{TRAIN_DATA}/*/*"))
keep = random.Random(0).sample(all_imgs, round(FRAC * len(all_imgs)))
shutil.rmtree(SUBSET_DATA, ignore_errors=True)
for p in keep:
    cls = os.path.basename(os.path.dirname(p))
    os.makedirs(f"{SUBSET_DATA}/{cls}", exist_ok=True)
    os.symlink(p, f"{SUBSET_DATA}/{cls}/{os.path.basename(p)}")
n_all = collections.Counter(p.split("/")[-2] for p in all_imgs)
n_keep = collections.Counter(p.split("/")[-2] for p in keep)
print(f"giữ {len(keep)}/{len(all_imgs)} ảnh →", ", ".join(f"{c} {n_keep[c] / n_all[c]:.0%}" for c in sorted(n_all)))
steps = -(-len(keep) // 512)   # train.py không drop_last → làm tròn lên
print("bước/epoch:", steps, "| tổng:", steps * EPOCHS, "(đủ dữ liệu 5 epoch: 95 × 5 = 475)")

giữ 12234/48938 ảnh → Angry 25%, Disgust 25%, Fear 25%, Happy 26%, Neutral 24%, Sad 25%, Surprise 25%
bước/epoch: 24 | tổng: 480 (đủ dữ liệu 5 epoch: 95 × 5 = 475)


# Train seed 1 trên tập con
Đúng cấu hình EXP-000 (runner.sh), chỉ đổi `--train-data` và `--epochs`. Bị ngắt thì chạy lại cell tạo tập con rồi cell này (tự resume).

In [7]:
hf_offline()
resume = f"{OUT_DIR}/{JOB_ID}-resume.pth"
run(f'/content/venv310/bin/python train.py '
    f'--workers 8 --epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
    f'--lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed {SEED} '
    f'--job-id {JOB_ID} --instruction "{INSTRUCTION}" --load-model CLIP_L14 '
    f'--train-data {SUBSET_DATA} --extra-checkpoint-dir {OUT_DIR}'
    + (f' --resume {resume}' if os.path.exists(resume) else ''))

Training date:  26-10-14 02:45
************************
workers = 8
epochs = 20
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 10
milestones = [30]
seed = 1
job_id = exp001_rand25_l14_seed1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp001/train
train_data = /content/CAER-S_rand25/train
sample_weights = None
resume = None
************************

  0%|                                               | 0.00/890M [00:00<?, ?iB/s]
  2%|▊                                      | 17.5M/890M [00:00<00:04, 183MiB/s]
  4%|█▌                                     | 36.9M/890M [00:00<00:04, 195MiB/s]
  6%|██▍                                    | 57.0M/890M [00:00<00:04, 202MiB/s]
  9%|███▎                                   | 76.3M/890M [

# Test zero-shot

In [8]:
!mkdir -p checkpoint log
if not os.path.exists(f"checkpoint/{JOB_ID}-model.pth"):
    !cp "$OUT_DIR/$JOB_ID-model.pth" checkpoint/
run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {JOB_ID} '
    f'--data-root /content/static_test_sets 2>&1 | tee log/{JOB_ID}-test.txt')
!cp log/$JOB_ID-test.txt "$OUT_DIR/"

************************
load_model = CLIP_L14
job_id = exp001_rand25_l14_seed1
data_root = /content/static_test_sets
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 50.21/56.49
************************* AffectNet7
UAR/WAR: 42.34/42.34
************************* AffectNet8
UAR/WAR: 36.77/36.77
************************* FERPlus
UAR/WAR: 38.28/45.54
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 56.85/63.95
************************* AffectNet7
UAR/WAR: 44.00/44.00
************************* AffectNet8
UAR/WAR: 38.30/38.30
************************* FERPlus
UAR/WAR: 40.12/53.09
****************************

# So sánh
Δ = lần này − mốc. Mốc chính: **EXP-000 seed1** (cùng seed, đủ dữ liệu). Δ gần 0 ở mọi dataset → head không nhạy với dữ liệu train.

In [9]:
import re, pandas as pd
REF = pd.DataFrame([{"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 50.36, "WAR": 56.58}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 42.51, "WAR": 42.51}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 36.95, "WAR": 36.95}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 38.27, "WAR": 45.45}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 56.73, "WAR": 64.05}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.0, "WAR": 44.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.33, "WAR": 38.33}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 39.99, "WAR": 52.92}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 54.94, "WAR": 65.12}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 43.97, "WAR": 43.97}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.02, "WAR": 38.02}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 36.63, "WAR": 48.89}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 58.88, "WAR": 66.88}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.09, "WAR": 44.09}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.23, "WAR": 39.23}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.65, "WAR": 51.47}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 57.73, "WAR": 65.91}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.94, "WAR": 44.94}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.55, "WAR": 39.55}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.45, "WAR": 49.29}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 50.23, "WAR": 54.56}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 41.47, "WAR": 41.47}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 35.76, "WAR": 35.76}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.58, "WAR": 45.82}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.34, "WAR": 62.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 43.79, "WAR": 43.79}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.4, "WAR": 38.4}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.89, "WAR": 51.62}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.64, "WAR": 64.78}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.39, "WAR": 44.39}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.08, "WAR": 38.08}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 37.95, "WAR": 49.17}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 58.07, "WAR": 65.32}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.17, "WAR": 44.17}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.84, "WAR": 38.84}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 41.55, "WAR": 53.41}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 57.19, "WAR": 64.6}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.97, "WAR": 44.97}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 39.45, "WAR": 39.45}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 42.17, "WAR": 49.46}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}])

rows, prompt, dataset = [], None, None
for line in open(f"log/{JOB_ID}-test.txt"):
    if "Prompt Type:" in line:
        prompt = line.split("Prompt Type:")[1].strip()
    elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
        dataset = line.strip().split()[-1]
    elif line.startswith("UAR/WAR:"):
        uar, war = map(float, line.split(":")[1].strip().split("/"))
        rows.append(dict(prompt=prompt, dataset=dataset, UAR=uar, WAR=war))
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/{JOB_ID}-results.csv", index=False)
# các lần EXP-001 trước (trọng số mềm, lọc cứng 25%)
prev = pd.concat([pd.read_csv(f"{OUT_DIR}/{j}-results.csv").assign(src=s) for s, j in CMP_JOBS.items()])
# paper chỉ báo 1 số/dataset (Ensemble Five) → bỏ ở các prompt khác
REF = REF[(REF.src != "paper") | (REF.prompt == "Expression Ensemble Five")]
REF = pd.concat([REF, prev[REF.columns]])

table = res.set_index(["prompt", "dataset"])
for src in ["EXP-000 seed1", *CMP_JOBS, "EXP-000 mean3", "paper"]:
    ref = REF[REF.src == src].set_index(["prompt", "dataset"])
    table[f"ΔUAR vs {src}"] = (table.UAR - ref.UAR).round(2)
table[f"ΔWAR vs EXP-000 seed1"] = (table.WAR - REF[REF.src == "EXP-000 seed1"].set_index(["prompt", "dataset"]).WAR).round(2)
pd.set_option("display.width", 250)
table

UAR    WAR  ΔUAR vs EXP-000 seed1  ΔUAR vs EXP-001 soft seed1  ΔUAR vs EXP-001 drop25 seed1  ΔUAR vs EXP-000 mean3  ΔUAR vs paper  ΔWAR vs EXP-000 seed1
prompt                                       dataset                                                                                                                                                               
Class Name                                   RAFDB       50.21  56.49                  -0.15                        0.00                         -0.03                  -0.02            NaN                  -0.09
                                             AffectNet7  42.34  42.34                  -0.17                       -0.26                         -0.20                   0.87            NaN                  -0.17
                                             AffectNet8  36.77  36.77                  -0.18                       -0.18                         -0.25                   1.01            NaN                  -0.18
                                             FERPlus     38.28  45.54                   0.01                        0.09                          0.11                  -0.30            NaN                   0.09
An Expression of Name                        RAFDB       56.85  63.95                   0.12                        0.26                          0.25                   1.51            NaN                  -0.10
                                             AffectNet7  44.00  44.00                   0.00                        0.03                         -0.06                   0.21            NaN                   0.00
                                             AffectNet8  38.30  38.30                  -0.03                       -0.03                         -0.03                  -0.10            NaN                  -0.03
                                             FERPlus     40.12  53.09                   0.13                        0.90                          1.00                   1.23            NaN                   0.17
A Photo of A Face with An Expression of Name RAFDB       55.08  65.22                   0.14                        0.20                          0.33                  -0.56            NaN                   0.10
                                             AffectNet7  44.23  44.23                   0.26                        0.20                          0.09                  -0.16            NaN                   0.26
                                             AffectNet8  38.30  38.30                   0.28                        0.25                          0.23                   0.22            NaN                   0.28
                                             FERPlus     36.65  49.01                   0.02                       -0.05                         -0.08                  -1.30            NaN                   0.12
Expression Ensemble Five                     RAFDB       58.95  66.75                   0.07                        0.48                          0.52                   0.88           0.25                  -0.13
                                             AffectNet7  44.29  44.29                   0.20                        0.23                          0.18                   0.12           0.02                   0.20
                                             AffectNet8  39.23  39.23                   0.00                        0.18                          0.18                   0.39           0.79                   0.00
                                             FERPlus     40.73  51.58                   0.08                        0.12                          0.15                  -0.82          -7.55                   0.11
Expression Ensemble Ten                      RAFDB       57.73  65.61                   0.00                       -0.11                          0.01                   0.54            NaN                  -0.30
                               

# Kiểm tra log train
So train accuracy cuối với lần đủ dữ liệu (27.6 ở epoch 5).

In [10]:
!grep -E "Sample weights|The best accuracy|An epoch time" log/$JOB_ID-log.txt

The best accuracy: 0.31419837474823
An epoch time: 130.58477401733398s
The best accuracy: 0.7727581858634949
An epoch time: 122.57533359527588s
The best accuracy: 1.4690897464752197
An epoch time: 122.74343538284302s
The best accuracy: 2.3607337474823
An epoch time: 122.49587678909302s
The best accuracy: 3.3542799949645996
An epoch time: 123.10802268981934s
The best accuracy: 4.449728488922119
An epoch time: 123.87922549247742s
The best accuracy: 5.647078990936279
An epoch time: 123.0397264957428s
The best accuracy: 7.073709487915039
An epoch time: 123.67040467262268s
The best accuracy: 9.001358985900879
An epoch time: 123.38272619247437s
The best accuracy: 10.784646987915039
An epoch time: 123.79986381530762s
The best accuracy: 12.151834487915039
An epoch time: 123.65802335739136s
The best accuracy: 14.50407600402832
An epoch time: 123.3547101020813s
The best accuracy: 16.8563175201416
An epoch time: 123.73241400718689s
The best accuracy: 18.588655471801758
An epoch time: 123.14779710